# Generation audio book with pdf file

In [ ]:
%pip install openai
%pip install python-dotenv
%pip install pydub
%pip install azure-core
%pip install azure-cognitiveservices-speech
%pip install azure-ai-documentintelligence

In [ ]:
# Add azure open ai endpoint
import os,re
from dotenv import load_dotenv

load_dotenv(dotenv_path='../keys.env')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_key = os.getenv('AZURE_OPENAI_KEY')
azure_speech_key = os.getenv('AZURE_SPEECH_KEY')
azure_speech_region = os.getenv('AZURE_SPEECH_REGION')
azure_document_intelligence_endpoint = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT')
azure_document_intelligence_key = os.getenv('AZURE_DOCUMENT_INTELLIGENCE_KEY')

folder_path = '../data/hachette'  # Adjust this path as needed

In [ ]:
pages2Process = "8-125"  # for new theory and Screen time is 7-316

In [ ]:

import azure.cognitiveservices.speech as speechsdk
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from azure.core.exceptions import ClientAuthenticationError
from pydub import AudioSegment

# Replace with your Document Intelligence endpoint and key
aoai_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
aoai_key = os.environ['AZURE_OPENAI_KEY']
# Replace with your Azure Speech Services subscription key and region
speech_key = os.environ['AZURE_SPEECH_KEY']
speech_region = os.environ['AZURE_SPEECH_REGION']
# Replace with your Document Intelligence endpoint and key
doc_endpoint = os.environ['AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT']
doc_key = os.environ['AZURE_DOCUMENT_INTELLIGENCE_KEY']


def analyze_pdf(pdf_path, directory, filename):
    """
    Analyzes a PDF document using Azure Document Intelligence and returns the text,
    along with page-level information.
    """
    try:
        document_intelligence_client = DocumentIntelligenceClient(
            endpoint=doc_endpoint, credential=AzureKeyCredential(doc_key)
        )
        with open(pdf_path, "rb") as f:
            document_content = f.read()
            poller = document_intelligence_client.begin_analyze_document(
                "prebuilt-layout",
                output_content_format=DocumentContentFormat.MARKDOWN,
                pages=pages2Process,
                body=document_content
            )
        result = poller.result()
        markdown_output = result.content

        full_text_file_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}_full_text.md")
        with open(full_text_file_path, "w", encoding="utf-8") as full_text_file:
            full_text_file.write(markdown_output)
        print(f"Full text written to {full_text_file_path}")

        return True, full_text_file_path

    except ClientAuthenticationError as e:
        print(f"Authentication error: {e}")
        print("Please check your Document Intelligence endpoint and key.")
        return None, None
    except Exception as e:
        print(f"Error analyzing {pdf_path}: {e}")
        return None, None



def split_markdown_by_chapters(input_file, output_dir):
    """
    Splits a Markdown file into separate chapter files based on chapter headings.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split the content by chapter headings
    chapters = re.split(r'(##\s*\d+)', content)

    # Ensure that the first item is not a chapter heading
    if chapters[0].strip() == '':
        chapters.pop(0)

    # Iterate through the chapters and write them to separate files
    for i in range(0, len(chapters), 2):
        chapter_heading = chapters[i].strip()
        chapter_content = chapters[i+1].strip() if i+1 < len(chapters) else ''
        # Create a valid filename from the chapter heading
        # Remove invalid characters from the chapter heading
        safe_chapter_heading = "".join(c if c.isalnum() or c in "._-" else "_" for c in chapter_heading)
        # Truncate the chapter heading to a reasonable length
        safe_chapter_heading = safe_chapter_heading[35]
        # Create a valid filename from the chapter heading
        chapter_filename = f"{i}_{safe_chapter_heading}.md"
        chapter_filepath = os.path.join(output_dir, chapter_filename)

        # Write the chapter content to the file
        with open(chapter_filepath, 'w', encoding='utf-8') as chapter_file:
            chapter_file.write(f"{chapter_heading}\n\n{chapter_content}")
        print(f"Chapter written to {chapter_filepath}")


def process_directory(directory):
    """
    Processes all PDF files in the given directory.
    """
    for filename in os.listdir(directory):
        if filename.endswith(".pdf"):
            pdf_path = os.path.join(directory, filename)
            # Analyze PDF and get full_text_file_path
            full_text_process, full_text_file_path = analyze_pdf(pdf_path, directory, filename)

            if full_text_process == True:
                input_file = full_text_file_path
                # Split the Markdown file by chapters
                output_dir = os.path.join(directory, os.path.splitext(filename)[0]+'_chapters')
                split_markdown_by_chapters(input_file, output_dir)

            else:
                print(f"Failed to extract text from {pdf_path}")

# launch the process
if __name__ == "__main__":
    # Define the root directory to scan
    root_directory = folder_path  # Use the folder_path variable
    # Iterate through all subdirectories in the root directory
    for subdir, dirs, files in os.walk(root_directory):
        print(f"Processing directory: {subdir}")
        process_directory(subdir)

Processing directory: ../hachette
Processing directory: ../hachette\ScreenTime
Full text written to ../hachette\ScreenTime\Screen Time - Lisa Guernsey_full_text.md
Chapter written to ../hachette\ScreenTime\Screen Time - Lisa Guernsey_chapters\0__.md
Error during language detection: module 'azure.cognitiveservices.speech.translation' has no attribute 'LanguageDetectionClient'
Failed to detect language, defaulting to 'en'
Speech synthesis canceled: CancellationReason.Error
Error details: USP error: timeout waiting for all remaining audio data. Received audio size 16766436 bytes.
Failed to generate audio for 0__.md
No audio files were created for this PDF.
Processing directory: ../hachette\ScreenTime\Screen Time - Lisa Guernsey_chapters
Processing directory: ../hachette\temp_audio
Processing directory: ../hachette\Theory of teenagers
Full text written to ../hachette\Theory of teenagers\A New Theory of Teenagers_ Seven Transform - Christa Santangelo_full_text.md
Chapter written to ../hache

## Generate audio book 

In [ ]:
temp_folder = folder_path+"/temp_audio"  # Temporary folder for page-level audio
# Create temporary folder if it doesn't exist
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)

In [ ]:
def process_chapter_audio(chapter_dir, language):
    """
    Processes each chapter file in the given directory to generate audio.
    """
    chapter_audio_files = []
    for filename in os.listdir(chapter_dir):
        if filename.endswith(".md"):
            chapter_file_path = os.path.join(chapter_dir, filename)
            try:
                with open(chapter_file_path, "r", encoding="utf-8") as chapter_file:
                    chapter_content = chapter_file.read()

                # Generate audio from the chapter content
                output_filename = f"{os.path.splitext(filename)[0]}.wav"
                output_path = os.path.join(chapter_dir, output_filename)
                if text_to_speech(chapter_content, language, output_path):
                    chapter_audio_files.append(output_path)
                else:
                    print(f"Failed to generate audio for {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    return chapter_audio_files

def text_to_speech(text, language ="en", output_path):
    """
    Converts text to speech using Azure Cognitive Services Speech SDK.
    """
    try:
        speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)

        # Set the voice based on the detected language (you might need to adjust voice names)
        if language == 'fr':
            speech_config.speech_synthesis_voice_name = "fr-FR-Vivienne:DragonHDLatestNeural"  # Example French voice
        elif language == 'en':
            speech_config.speech_synthesis_voice_name = "en-US-Ava3:DragonHDLatestNeural"  # Example English voice
        else:
            speech_config.speech_synthesis_voice_name = "en-US-Andrew3:DragonHDLatestNeural"  # Default to English

        # Initialize the speech synthesizer
        speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=None)

        # Synthesize the text to speech
        result = speech_synthesizer.speak_text_async(text).get()

        # Check result
        if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
            # Save the audio to a WAV file
            with open(output_path, "wb") as audio_file:
                audio_file.write(result.audio_data)
            print(f"Speech synthesized for {output_path}")
            return True
        elif result.reason == speechsdk.ResultReason.Canceled:
            cancellation_details = result.cancellation_details
            print(f"Speech synthesis canceled: {cancellation_details.reason}")
            if cancellation_details.reason == speechsdk.CancellationReason.Error:
                print(f"Error details: {cancellation_details.error_details}")
            return False
    except Exception as e:
        print(f"Error during speech synthesis: {e}")
        return False


 with open(input_file, 'r', encoding='utf-8') as f:
    content = f.read()
    # Process audio for each chapter
    chapter_audio_files = process_chapter_audio(output_dir, language)

In [ ]:

def combine_audio_files(input_files, output_file):
    """
    Combines multiple audio files into a single audio file.
    """
    combined_audio = AudioSegment.empty()
    for file in input_files:
        audio = AudioSegment.from_wav(file)
        combined_audio += audio

    combined_audio.export(output_file, format="wav")
    print(f"Combined audio saved to {output_file}")

# Combine the audio files for all chapters
if chapter_audio_files:
    final_output_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}_combined_chapters.wav")
    combine_audio_files(chapter_audio_files, final_output_path)

    # Clean up temporary audio files
    for file in chapter_audio_files:
        os.remove(file)
    print(f"Temporary audio files removed from {temp_folder}")
else:
    print("No audio files were created for this PDF.")